In [4]:
from multiprocessing import freeze_support

import os
import sys
import pickle
import pprint
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

sys.path.append("../")

import torch as t
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

from tqdm import tqdm
from datetime import datetime
import json
import wandb

from devinterp.slt import estimate_learning_coeff, estimate_learning_coeff_with_summary
from devinterp.optim.sgld import SGLD
from devinterp.slt import sample
from devinterp.slt.llc import OnlineLLCEstimator
from devinterp.slt.wbic import OnlineWBICEstimator

from approxngd import KFAC
from PyHessian.pyhessian import *
from PyHessian.density_plot import *
from general_utils import *
from hessian_utils import *
from architectures.NN import *
from data.build_data import build_data

import plotly.express as px
import plotly.graph_objects as go
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

device = "cuda" if t.cuda.is_available() else "cpu"
print(device)

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
%load_ext autoreload
%autoreload
%matplotlib inline

In [5]:
# vary optimisers
optimisers = ["ngd", "sgd"]
models = {}
for optim in optimisers:
    models[optim] = LinearMNIST(hidden_nodes=1024, hidden_layers=2).to(device)

In [8]:
hyperparams = {
    "batch_size" : 512,
    "num_workers" : 12,
    "num_epochs" : 10,
    "lr" : 9e-03,
    "momentum" : 0.9,
}

In [7]:
# load data from pytorch dataloaders

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = t.utils.data.DataLoader(train_set, batch_size=hyperparams["batch_size"], shuffle=True, num_workers=hyperparams["num_workers"], persistent_workers=True)
test_loader = t.utils.data.DataLoader(test_set, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=hyperparams["num_workers"], persistent_workers=True)

In [9]:
# training loop (vary optimisers)

train_losses = {}
test_losses = {}
for optim, model in models.items():
    print(f"TRAINING WITH {optim} OPTIMISER")
    train_losses[optim] = []
    test_losses[optim] = []
    sgd = t.optim.SGD(model.parameters(), lr=hyperparams["lr"], momentum=hyperparams["momentum"], nesterov=True)
    ngd = KFAC(model, hyperparams["lr"], damping=0.01, momentum_type='regular', momentum=0.8, adapt_damping=False, update_cov_manually=False)
    optimiser = ngd if optim=="ngd" else sgd
    criterion = nn.CrossEntropyLoss(reduction='mean') if optim=="ngd" else nn.CrossEntropyLoss()
    for epoch in range(1, hyperparams["num_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimiser, criterion, device)
        test_loss = evaluate(model, test_loader, criterion, device)
        train_losses[optim].append(train_loss)
        test_losses[optim].append(test_loss)
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}: train_loss={train_loss:.4f}, test_loss={test_loss:.4f}")

TRAINING WITH ngd OPTIMISER


  0%|          | 0/118 [00:00<?, ?it/s]c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\torch\nn\modules\module.py:1352: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "
c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\approxngd\utils\utils.py:102: UserWarning: torch.cholesky is deprecated in favor of torch.linalg.cholesky and will be removed in a future PyTorch release.
L = torch.cholesky(A)
should be replaced with
L = torch.linalg.cholesky(A)
and
U = torch.cholesky(A, upper=True)
should be replaced with
U = torch.linalg.cholesky(A).mH
This transform will produce equivalent results for all valid (symmetric positive definite) inputs. (Triggered internally at 

Epoch 2/10: train_loss=0.4355, test_loss=0.2876


100%|██████████| 118/118 [00:38<00:00,  3.05it/s]


Epoch 3/10: train_loss=0.2855, test_loss=0.2820


...

Epoch 8/10: train_loss=0.2728, test_loss=0.2802


100%|██████████| 118/118 [00:01<00:00, 84.80it/s]


Epoch 9/10: train_loss=0.2699, test_loss=0.2784


100%|██████████| 118/118 [00:01<00:00, 82.25it/s]


Epoch 10/10: train_loss=0.2668, test_loss=0.2771


In [10]:
# visualise training and testing data
epochs = np.arange(1, hyperparams["num_epochs"]+1)
train_fig = go.Figure()
for optim, train_loss in train_losses.items():
    train_fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode='lines+markers', name=optim))

train_fig.update_layout(title="Training loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Optimiser"
                  )
train_fig.show()

test_fig = go.Figure()
for optim, test_loss in test_losses.items():
    test_fig.add_trace(go.Scatter(x=epochs, y=test_loss, mode='lines+markers', name=optim))

test_fig.update_layout(title="Test loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Optimiser",
                  )
test_fig.show()

In [32]:
# perform LLC estimation at convergence

NUM_CHAINS = 1
NUM_DRAWS = 500

optim_kwargs = dict(
    lr=1e-06,
    elasticity=1,
    num_samples=len(train_set),
    temperature="adaptive",
)

models_results = {}
for optim, model in models.items():
    results = estimate_learning_coeff_with_summary(
        model=model,
        loader=train_loader,
        sampling_method=SGLD,
        criterion=criterion,
        optimizer_kwargs=optim_kwargs,
        num_chains=NUM_CHAINS,
        num_draws=NUM_DRAWS,
        num_steps_bw_draws=1,
        num_burnin_steps=0,
        device=device,
        online=True,
    )
    models_results[optim] = results
    
LLC_fig = go.Figure()
for optim, results in models_results.items():
    LLC_fig.add_trace(go.Scatter(
        x=np.arange(1, NUM_DRAWS+1),
        y=results['llc/means'],
        mode='lines',
        name=optim,
    ))
    LLC_fig.update_layout(title="LLC estimation evolution",
                  xaxis_title="Draws",
                  yaxis_title="LLC",
                  legend_title="Optimiser",
                  )
LLC_fig.show()

Chain 0: 100%|██████████| 500/500 [00:02<00:00, 185.61it/s]


In [17]:
NUM_EXPERIMENTS = 5
RESULTS = np.zeros((NUM_EXPERIMENTS, 2))

for i in range(NUM_EXPERIMENTS):
    for optim, model in models.items():
        results = estimate_learning_coeff_with_summary(
            model=model,
            loader=train_loader,
            sampling_method=SGLD,
            criterion=criterion,
            optimizer_kwargs=optim_kwargs,
            num_chains=NUM_CHAINS,
            num_draws=NUM_DRAWS,
            num_steps_bw_draws=1,
            num_burnin_steps=0,
            device=device,
            online=True,
        )
        if optim == "ngd":
            RESULTS[i, 0] = results['llc/means'][-1]
        else:
            RESULTS[i, 1] = results['llc/means'][-1]

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\devinterp\slt\sampler.py:46: UserWarning:

You are taking more sample batches than there are dataloader batches available, this removes some randomness from sampling but is probably fine. (All sample batches beyond the number dataloader batches are cycled from the start, f.e. 9 samples from [A, B, C] would be [B, A, C, B, A, C, B, A, C].)

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\devinterp\optim\sgld.py:71: UserWarning:


Chain 0:   0%|          | 0/500 [00:00<?, ?it/s]c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\torch\nn\modules\module.py:1352: UserWarning:

Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.

Chain 4: 100%|██████████| 500/500 [00:03<00:00, 151.37it/s]


TypeError: sum() got an unexpected keyword argument 'dim'

In [18]:
final_results = (RESULTS[:, 0] > RESULTS[:, 1]).astype(int)
perc = np.sum(final_results) * (100 / NUM_EXPERIMENTS)
print(f"Percentage where NGD had higher LLC: {perc}")

Percentage where NGD had higher LLC: 40.0


In [20]:
RESULTS

array([[ 96.27579498,  65.80242157],
       [ 58.77090073,  73.87365723],
       [-21.47498131,  15.18047523],
       [-21.69681358, -46.05247879],
       [-75.66129303, 115.0654068 ]])